# Denser logit-space grid, plausibility constraint and decoding throughput

**GPU: NVIDIA A100.** Run after `colab_run.ipynb`, `colab_gsm8k_budget.ipynb` and `colab_kl_seeds.ipynb`. No adapter is trained here.

| Stage | Content |
|---|---|
| 1 | Logit-space factors 0.8, 0.85, 0.9 and 0.95 for the fully trained adapters: own-task score and KL divergence to the base model, so that the comparison at equal divergence interpolates between closer grid points. The multiple-choice tasks first (one pair of passes serves all factors), then the generation tasks; GSM8K answers cut by the token budget are then decoded again with 768 tokens |
| 2 | Logit-space amplification with the plausibility constraint of contrastive decoding (threshold 0.1) on the generation tasks, factors 1.05 to 2 |
| 3 | Decoding throughput of the base model, the adapter, weight-space scaling (active and merged) and logit-space scaling |

Set `STAGES` in the configuration cell to run a subset. The configuration cell prints an estimate of the run time from the times already recorded on Drive. **Everything is resumable**: after a disconnection, run all cells again; finished factors and files are skipped. Then run the last cell (or `colab_analysis.ipynb`).

## 0. Setup

In [ ]:
import os, subprocess, sys
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', GPU or 'none')
assert GPU, 'No GPU: Runtime > Change runtime type'
EXPECTED = ('A100',)
ALLOW_OTHER_GPU = False   # set to True to run on another GPU anyway (it is recorded in run_info.json)
if not any(g in GPU for g in EXPECTED) and not ALLOW_OTHER_GPU:
    raise SystemExit(f'This notebook is planned for {EXPECTED} (bfloat16, 40 GB or more), got {GPU}: Runtime > Change runtime type')

REPO = '/content/dynalora'
BRANCH = 'extra'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} https://github.com/alexroehrs/dynalora.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log -1 --format='commit %H  %s'
!pip install -q -r requirements.txt
# Colab ships torchao 0.10, and peft 0.21 then refuses to load any LoRA adapter (it requires torchao > 0.16).
# Nothing in this repository uses torchao, so it is removed.
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/dynalora_results'   # results root (Google Drive); results are not kept in the repository
L4 = f'{RESULTS}/level4'
os.makedirs(L4, exist_ok=True)
sys.path.insert(0, REPO)
from dynalora.runinfo import record_session
e = record_session(f'{L4}/run_info.json', 'extra notebook start', seeds={'adapter_training': 'per adapter (s<seed> in its name)', 'train_subsets': 42, 'test_subsets': 0, 'bootstrap': 0})
print(e['gpu'], e.get('gpu_memory_gb'), e['git'], e['libraries'], sep='\n')
import torch
assert torch.cuda.is_bf16_supported(), 'this GPU has no bfloat16 support'
FAILED = []   # steps that did not finish; reported at the end

CPU tests of the new code paths (random tiny models, about 1 min).

In [ ]:
!python -m pytest -q tests/test_tasks.py -k "plausibility or without_stop or greedy"

## Configuration

In [ ]:
import glob, pandas as pd

def sh(cmd, log):
    print(cmd)
    get_ipython().system(f"bash -c 'set -o pipefail; {cmd} 2>&1 | tee -a {log}'")
    return get_ipython().user_ns.get('_exit_code', 1)

def step(name, cmd):
    rc = sh(cmd, f'{L4}/extra.log')
    if rc != 0:
        FAILED.append(f'{name} (exit code {rc})'); print(f'WARNING: {name} did not finish; rerun the notebook to resume')
    return rc == 0

STAGES = [1, 2, 3]                       # remove a number to skip a stage
PRIMARY, SECOND = 'Qwen/Qwen2.5-7B-Instruct', 'microsoft/Phi-4-mini-instruct'
CHOICE, GEN = ['pubmedqa', 'yelp5'], ['gsm8k', 'samsum']
ADAPTERS = [(PRIMARY, 42), (SECOND, 42), (PRIMARY, 43), (PRIMARY, 44)]   # in this order: seed 42 of both models first
DENSE = '0.8 0.85 0.9 0.95'              # stage 1: logit-space factors added between 0.75 and 1
PLAUS = '1.05 1.1 1.15 1.25 1.5 2'       # stage 2: logit-space factors with the plausibility constraint
PLAUS_ADAPTERS = [(PRIMARY, 42), (SECOND, 42)]   # add (PRIMARY, 43), (PRIMARY, 44) to cover the three seeds
ALPHA, KL_ITEMS, BUDGET = 0.1, 200, 768
A_DIR, S_DIR, T_DIR = f'{L4}/adapters', f'{L4}/scores', f'{L4}/throughput'
tag = lambda m: m.rstrip('/').split('/')[-1]
name = lambda m, t, s: f'{tag(m)}__{t}__s{s}'

def eval_bs(model, task):
    small = 'mini' in model.lower()
    return (32 if small else 16) if task in GEN else (16 if small else 8)

def have(model, task, seed):
    ok = os.path.isdir(f'{A_DIR}/{name(model, task, seed)}') and os.path.exists(f'{S_DIR}/{name(model, task, seed)}/own__logit.csv')
    if not ok:
        FAILED.append(f'{name(model, task, seed)}: adapter or scores not found'); print(FAILED[-1])
    return ok

# Estimate from the times recorded by the earlier notebooks (seconds of one logit-space decoding of 500 items).
est = {}
for model, seed in ADAPTERS:
    for t in GEN:
        f = f'{S_DIR}/{name(model, t, seed)}/timing.csv'
        if os.path.exists(f):
            tm_ = pd.read_csv(f); tm_ = tm_[(tm_.what == 'score') & (tm_.form == 'logit') & (tm_.task == t)]
            tm_['factor'] = pd.to_numeric(tm_.factor, errors='coerce')
            lo, hi = tm_[tm_.factor == 0.75].seconds.max(), tm_[tm_.factor.between(1.05, 2)].seconds.mean()
            est[(tag(model), seed, t)] = (4 * lo / 3600 if lo == lo else None, 6 * hi / 3600 if hi == hi else None)
if est:
    e = pd.DataFrame([dict(model=k[0], seed=k[1], task=k[2], stage1_hours=v[0], stage2_hours=v[1]) for k, v in est.items()])
    e.loc[~e.apply(lambda r: any(tag(m) == r.model and s == r.seed for m, s in PLAUS_ADAPTERS), axis=1), 'stage2_hours'] = 0
    print(e.round(2).to_string(index=False))
    print(f'estimated total: stage 1 about {e.stage1_hours.sum():.1f} h, stage 2 about {e.stage2_hours.sum():.1f} h, stage 3 a few minutes')

## Stage 1. Logit-space factors between 0.75 and 1

In [ ]:
def dense(model, task, seed):
    n = name(model, task, seed)
    return step(f'dense {n}', f'python scripts/eval_task.py --model {model} --task {task} --adapter_dir {A_DIR}/{n} --out_dir {S_DIR}/{n} '
                f'--base_cache {S_DIR}/_base --batch_size {eval_bs(model, task)} --forms logit --lambdas {DENSE} --kl_items {KL_ITEMS}')

def extend(model, seed):
    n = name(model, 'gsm8k', seed)
    return step(f'extend {n}', f'python scripts/extend_cut_outputs.py --model {model} --task gsm8k --base_cache {S_DIR}/_base '
                f'--max_new_tokens {BUDGET} --batch_size {32 if "mini" in model.lower() else 16} --adapter_dir {A_DIR}/{n} --out_dir {S_DIR}/{n}')

if 1 in STAGES:
    for tasks in (CHOICE, GEN):                      # multiple choice first (minutes), then generation
        for model, seed in ADAPTERS:
            for t in tasks:
                if have(model, t, seed) and dense(model, t, seed) and t == 'gsm8k':
                    extend(model, seed)
print('not finished:', FAILED or 'nothing')

## Stage 2. Logit-space amplification with the plausibility constraint

In [ ]:
if 2 in STAGES:
    for model, seed in PLAUS_ADAPTERS:
        for t in GEN:
            n = name(model, t, seed)
            if have(model, t, seed):
                step(f'plausibility {n}', f'python scripts/eval_plausibility.py --model {model} --task {t} --adapter_dir {A_DIR}/{n} '
                     f'--out_dir {S_DIR}/{n} --alpha {ALPHA} --lambdas {PLAUS} --batch_size {eval_bs(model, t)}')
print('not finished:', FAILED or 'nothing')

## Stage 3. Decoding throughput

In [ ]:
if 3 in STAGES:
    os.makedirs(T_DIR, exist_ok=True)
    for model in (PRIMARY, SECOND):
        n = name(model, 'samsum', 42)
        if have(model, 'samsum', 42):
            step(f'throughput {tag(model)}', f'python scripts/throughput.py --model {model} --task samsum --adapter_dir {A_DIR}/{n} '
                 f'--out {T_DIR}/{tag(model)}.csv')
print('not finished:', FAILED or 'nothing')

## Analysis

In [ ]:
!python scripts/analysis/summarize.py --results {RESULTS} > {L4}/analysis_extra.log 2>&1; tail -n 5 {L4}/analysis_extra.log
A = f'{L4}/analysis'
pd.set_option('display.width', 250)
eq = pd.read_csv(f'{A}/equal_divergence.csv').dropna(subset=['factor_logit'])
eq['within2'] = eq.weight_minus_logit.abs() <= 0.02
eq['ci_within2'] = (eq.diff_lo >= -0.02) & (eq.diff_hi <= 0.02)
eq['significant'] = (eq.diff_lo > 0) | (eq.diff_hi < 0)
print('== equal divergence:', len(eq), 'comparisons;', int(eq.within2.sum()), 'within two points;', int(eq.ci_within2.sum()),
      'with the interval within two points;', int(eq.significant.sum()), 'significant; median |difference|', round(100 * eq.weight_minus_logit.abs().median(), 2))
print(eq.groupby('model')[['within2', 'ci_within2', 'significant']].agg(['sum', 'count']).to_string())
for f in ('plausibility', 'throughput'):
    if os.path.exists(f'{A}/{f}.csv'):
        print(f'== {f}'); print(pd.read_csv(f'{A}/{f}.csv').round(4).to_string(index=False))
print('not finished:', FAILED or 'nothing')